# 01 - Exploratory Data Analysis (EDA)

**Dynamic Pricing Engine** - Inside Airbnb NYC Dataset

This notebook explores:
1. Dataset shapes, dtypes, and missing values
2. Price distribution analysis
3. Temporal patterns in bookings
4. Room type breakdown
5. Neighborhood analysis
6. Correlation analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.ingestion import load_airbnb_data
from src.data.preprocessing import clean_price_column
from src.utils.config import load_config

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='viridis')

config = load_config()
print('Config loaded. City:', config['data']['city'])

## 1. Load Data

In [ ]:
# Load all datasets (downloads automatically if not present)
datasets = load_airbnb_data(config)

listings = datasets['listings']
calendar = datasets['calendar']
reviews = datasets['reviews']

print(f'Listings: {listings.shape}')
print(f'Calendar: {calendar.shape}')
print(f'Reviews:  {reviews.shape}')

## 2. Dataset Overview

In [ ]:
# Listings: dtypes and missing values
print('=== LISTINGS ===')
print(f'Shape: {listings.shape}')
print(f'\nDtypes:\n{listings.dtypes.value_counts()}')
print(f'\nMissing Values (top 15):')
missing = (listings.isnull().sum() / len(listings) * 100).sort_values(ascending=False)
print(missing[missing > 0].head(15).to_string())

In [ ]:
# Calendar overview
print('=== CALENDAR ===')
print(f'Shape: {calendar.shape}')
print(f'Date range: {calendar["date"].min()} to {calendar["date"].max()}')
print(f'\nAvailability:\n{calendar["available"].value_counts(normalize=True)}')

In [ ]:
# Reviews overview
print('=== REVIEWS ===')
print(f'Shape: {reviews.shape}')
reviews['date'] = pd.to_datetime(reviews['date'])
print(f'Date range: {reviews["date"].min()} to {reviews["date"].max()}')
print(f'Unique listings reviewed: {reviews["listing_id"].nunique():,}')

## 3. Price Distribution

In [ ]:
# Clean price column
listings['price_clean'] = clean_price_column(listings['price'])

print('Price Statistics (before outlier removal):')
print(listings['price_clean'].describe())

# Clip outliers for visualization
p5 = listings['price_clean'].quantile(0.05)
p95 = listings['price_clean'].quantile(0.95)
price_clipped = listings['price_clean'][(listings['price_clean'] >= p5) & (listings['price_clean'] <= p95)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(price_clipped, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Price Distribution (5th-95th percentile)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(price_clipped), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title('Log Price Distribution')
axes[1].set_xlabel('Log(Price + 1)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Room Type Analysis

In [ ]:
# Room type distribution
room_counts = listings['room_type'].value_counts()
print('Room Type Distribution:')
print(room_counts)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'box'}]])

fig.add_trace(
    go.Pie(labels=room_counts.index, values=room_counts.values, hole=0.4),
    row=1, col=1
)

for rt in listings['room_type'].unique():
    prices = listings[listings['room_type'] == rt]['price_clean']
    prices = prices[(prices >= p5) & (prices <= p95)]
    fig.add_trace(
        go.Box(y=prices, name=rt),
        row=1, col=2
    )

fig.update_layout(title='Room Type: Distribution & Price Comparison', height=450)
fig.show()

## 5. Neighborhood Analysis

In [ ]:
# Top neighborhoods by listing count and median price
if 'neighbourhood_group_cleansed' in listings.columns:
    borough_stats = listings.groupby('neighbourhood_group_cleansed').agg(
        count=('id', 'count'),
        median_price=('price_clean', 'median'),
        mean_price=('price_clean', 'mean'),
        avg_reviews=('number_of_reviews', 'mean'),
    ).sort_values('count', ascending=False)
    
    print('Borough-Level Stats:')
    print(borough_stats.to_string())
    
    fig = px.bar(
        borough_stats.reset_index(),
        x='neighbourhood_group_cleansed',
        y='median_price',
        color='count',
        title='Median Price by Borough (color = listing count)',
        labels={'neighbourhood_group_cleansed': 'Borough', 'median_price': 'Median Price ($)'}
    )
    fig.show()

In [ ]:
# Geographic scatter plot
sample = listings.sample(min(5000, len(listings)), random_state=42)
fig = px.scatter_mapbox(
    sample,
    lat='latitude',
    lon='longitude',
    color='price_clean',
    size='number_of_reviews',
    color_continuous_scale='Viridis',
    size_max=10,
    zoom=10,
    title='NYC Airbnb Listings - Price Heatmap',
    mapbox_style='carto-positron',
    labels={'price_clean': 'Price ($)'},
    range_color=[p5, p95],
)
fig.update_layout(height=600)
fig.show()

## 6. Temporal Patterns (Calendar)

In [ ]:
# Booking rate over time
cal = calendar.copy()
cal['date'] = pd.to_datetime(cal['date'])
cal['was_booked'] = (cal['available'] == 'f').astype(int)
cal['day_of_week'] = cal['date'].dt.day_name()
cal['month'] = cal['date'].dt.month

# Daily booking rate
daily_rate = cal.groupby('date')['was_booked'].mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(daily_rate.index, daily_rate.values, linewidth=0.8)
axes[0].set_title('Daily Booking Rate Over Time')
axes[0].set_ylabel('Booking Rate')

# Day-of-week pattern
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_rate = cal.groupby('day_of_week')['was_booked'].mean().reindex(dow_order)
axes[1].bar(dow_order, dow_rate.values, color='steelblue', edgecolor='black')
axes[1].set_title('Booking Rate by Day of Week')
axes[1].set_ylabel('Booking Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix of numeric features
numeric_cols = ['price_clean', 'minimum_nights', 'number_of_reviews',
                'reviews_per_month', 'availability_365', 'latitude', 'longitude']
numeric_cols = [c for c in numeric_cols if c in listings.columns]

corr = listings[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Feature Correlation Matrix',
    aspect='auto',
)
fig.update_layout(height=500)
fig.show()

## 8. Review Velocity as Demand Proxy

In [ ]:
# Review velocity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

review_vel = listings['reviews_per_month'].dropna()
review_vel = review_vel[review_vel > 0]

axes[0].hist(review_vel, bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0].set_title('Reviews per Month (Demand Proxy)')
axes[0].set_xlabel('Reviews / Month')

# Price vs review velocity
sample = listings[(listings['reviews_per_month'] > 0) & (listings['price_clean'] >= p5) & (listings['price_clean'] <= p95)]
axes[1].scatter(sample['reviews_per_month'], sample['price_clean'], alpha=0.1, s=5)
axes[1].set_title('Price vs Review Velocity')
axes[1].set_xlabel('Reviews / Month')
axes[1].set_ylabel('Price ($)')
plt.tight_layout()
plt.show()

print(f'Correlation (price vs reviews_per_month): {listings["price_clean"].corr(listings["reviews_per_month"]):.4f}')

## 9. Key Takeaways

Fill in after running the notebook:

- **Dataset Size**: ___ listings, ___ calendar entries, ___ reviews
- **Price Range**: Median $___,  5th-95th: $___-$___
- **Most common room type**: ___
- **Highest-priced borough**: ___
- **Weekend booking premium**: ___ vs weekday
- **Key correlations to price**: ___